# 02 | EDA and target definition

**Goal:** examine the target and its relationship to geometry and wind direction, then fix the target definition.

`U_canopy_mean = (1 / h_mean) * integral_0^h_mean U(z) dz` using trapezoidal integration with linear interpolation at the endpoints. Every case has the same mean building height (16 m).

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 30)
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed'; RESULTS = ROOT / 'results'
import sys; sys.path.insert(0, str(ROOT / 'src'))
df = pd.read_csv(DATA / 'layout_level_dataset.csv')
target = 'target_U_canopy_mean_m_s'
print(df[target].describe().round(3).to_string())
print('\nBy wind direction:'); print(df.groupby('wind_direction_deg')[target].agg(['count','mean','std','min','max']).round(3).to_string())
print('\nBy arrangement:'); print(df.groupby('horizontal_configuration')[target].agg(['count','mean','std']).round(3).to_string())
print('\nBy height pattern:'); print(df.groupby('configuration')[target].agg(['count','mean','std']).round(3).to_string())

count    224.000
mean       2.082
std        1.002
min        0.966
25%        1.390
50%        1.630
75%        2.472
max        5.275

By wind direction:
                    count   mean    std    min    max
wind_direction_deg                                   
0.0                   112  2.276  1.194  0.966  5.275
45.0                  112  1.889  0.720  1.111  3.696

By arrangement:
                          count   mean    std
horizontal_configuration                     
Aligned                     112  2.272  1.041
Staggered                   112  1.893  0.928

By height pattern:
                   count   mean    std
configuration                         
idealized-CL          64  2.086  0.986
idealized-HR          64  2.047  1.010
idealized-SM          64  2.054  0.990
idealized-uniform     32  2.201  1.080


## Density dependence

Mean canopy speed against plan-area density for each arrangement and direction. The decay is steep at low density and flattens at high density (convex), and the 0° / 45° gap shrinks as density rises. It is not strictly monotonic: the staggered 45° column ticks up slightly (about +0.04 m/s) between densities 0.35 and 0.44, which is small next to the overall drop. This is the *canopy-mean* target; a pedestrian-level quantity could behave differently.

In [2]:
tab = df.pivot_table(index='plan_area_density', columns=['horizontal_configuration','wind_direction_deg'], values=target, aggfunc='mean').round(2)
print(tab.to_string())
print('\nColumns with at least one increase as density rises:', [c for c in tab.columns if (tab[c].diff().dropna() > 0).any()])

horizontal_configuration Aligned       Staggered      
wind_direction_deg          0.0   45.0      0.0   45.0
plan_area_density                                     
0.062500                    5.12  3.42      4.22  3.64
0.111111                    3.71  2.37      2.87  2.48
0.160000                    2.98  1.94      2.17  1.94
0.250000                    2.34  1.64      1.48  1.44
0.351166                    1.98  1.55      1.25  1.35
0.444444                    1.76  1.49      1.17  1.39
0.528926                    1.66  1.45      1.11  1.37
0.640000                    1.50  1.42      1.08  1.34

Columns with at least one increase as density rises: [('Staggered', 45.0)]


## EDA figure

![EDA overview](../reports/eda_overview.png)

Height-pattern statistics are nearly constant across families, so density and wind direction dominate the target. The grouped validation in notebook 04 asks whether a model can still predict a *whole unseen density/arrangement*. There are only two wind directions; predictions for other directions need new data.